# Lab03J — Liquidez condicional e verificação dos pareceres

**Único notebook novo a executar: CPU, executar tudo, salvar a cópia executada e o ZIP no Drive.** Não repita I/H/F/G/E. Este lote reutiliza o export I já conferido e os onze caches B3 de2015–2025; não consulta provedores nem baixa novamente cotações.

Executa os oito subconjuntos de controles (porte/setor/beta), duas coortes e três blocos; mudança de liquidez,84 diagnósticos das simulações existentes,83 disposições de eventos e três continuidades documentadas. Preserva80/117, ativo/classe/ISIN, dez setores, pisos e lacunas. A continuidade é apenas sensibilidade, não alteração da matriz principal.

O porte é capitalização aproximada de quantidade divulgada até2015, não inventário exato/free float. Inferência exploratória sem calibração específica. Não estabelece causalidade, equivalência ou homologação econômica global. Os modelos completos não rejeitam após Holm16; todos os resultados são mantidos.


In [ ]:
import os,sys,json,hashlib,zipfile,subprocess,shutil,importlib.util
from pathlib import Path
from datetime import datetime,timezone
from urllib.request import urlretrieve
for module,package in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('matplotlib','matplotlib'),('threadpoolctl','threadpoolctl')]:
    if importlib.util.find_spec(module) is None:subprocess.run([sys.executable,'-m','pip','install','--quiet',package],check=True)
import numpy as np,pandas as pd
from IPython.display import display,Image
LOCAL_MODE=bool(os.environ.get('ALEM_BETA_LOCAL'))
BASE=Path(os.environ.get('ALEM_BETA_LOCAL','/content/alem_beta_lab03j')).resolve();BASE.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ');RUN=BASE/('run_'+STAMP);RUN.mkdir();ROOT=RUN/'runtime';ROOT.mkdir()
def digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def script(name,*args):subprocess.run([sys.executable,str(ROOT/'scripts'/name),*map(str,args)],check=True)
print('Lab03J | CPU |','integração local, não Colab do autor' if LOCAL_MODE else 'Colab','|',RUN.name)


## Autenticação dos insumos existentes

Código e tabelas têm commit imutável e SHA256 por arquivo. O export I é lido como dados, sem executar os scripts recebidos. Caches no Drive são procurados e autenticados antes de qualquer cópia; eventual transferência usa arquivos já existentes no próprio Drive. Nenhuma consulta Yahoo/B3/CVM é necessária neste lote.


In [ ]:
SOURCE_COMMIT='657627b0addc16d8df3369dad9bcee0f1dc1c0da'
SOURCE_PINS={'data/lab03i/b3_cache_manifest.json': 'e9231565de3a81ea3a4586403ad55f50a5572442a69d4784546be2de28c675b8', 'data/lab03i/review_reference/universo_2015.csv': '319ba3c0faae6d9488b9a3fd58fbbbefd0cdd73c8492c826f888ad5536a9f595', 'data/lab03j/adendo_fontes_tamanho_v1.json': '4b55271aac32e31821fce8c98efd57d37d1f7df50b4bf67cea080c094d544bd9', 'data/lab03j/adendo_recuperacao_v2.json': '871400e43e6b0f224f63b1986a6a7ff773675b60967929c094a59b2a7beeb25b', 'data/lab03j/adendo_sucessoes_v1.json': '27b75e3559bea1d6d12570aa06020ac16656f9e4cc874b6794584030819bed0b', 'data/lab03j/batch_validation.json': 'f0579cd6ab60a6667b2b3018d00732599defd6ec52f79e7c852466dc9da5ee2f', 'data/lab03j/covariates_2015.csv': '177e2e31f410a6f1c4277ba62bdbdb4f575c2e01a386086b3801d776e1d5122e', 'data/lab03j/cvm_selected_members.zip': '988662919aa9f216a30186bc59337159d42fcee04279fcdc82107dd73ee9a59a', 'data/lab03j/cvm_sources_manifest.json': 'c2f6662ecdc6a5b950782d155bb396b6e4cafd0e8f2617c2040bd27e5a55a314', 'data/lab03j/issuer_map.csv': '2314e2ab20b71eeedbc015b76799d0eaa89e6b61a79d2f95a4f02b1ec908c344', 'data/lab03j/plano_v1.json': 'c9cf4677751485666bf9ed8f4088483ab8a5f00c22f80c461c5323efc9dbbff2', 'data/lab03j/size_source_audit.csv': 'df67b6fc409b25e45cae3a804fc29d6f05307796b59f5633542e66f9e1d865b0', 'data/lab03j/source_reading_ledger.json': '66819625bf793cda55d8d657827e089302a510fe421e9b29f1991c821f49ff2b', 'data/lab03j/test_validation.json': '78e91b34324584d5f9326a825b1736b21fa869d052bed5d20a44940bdecc3944', 'results/tables/lab03f_pos_parecer_20261024/aggregate.csv.gz': 'e4f1a4ff05ffc17194e826cd412eb651fe159a3055845cdcc62d7e08569e001a', 'results/tables/lab03g_pos_parecer_v3_20261024/calibration_cases.csv': '35e7263855f1c8d522e5d96ea4de3264a82f368843f6d16ba46149423f55dfc6', 'results/tables/lab03g_pos_parecer_v3_20261024/calibration_replications.csv.gz': 'cb6537858edd89e4517c3bf41e1e08d0dbb410c08f10ecdc0adb1e33594f9a61', 'results/tables/lab03g_pos_parecer_v3_20261024/observed_aggregate.csv.gz': 'c1b448523e4c0e86fc1d0aba3f2dbebbb4defa2878d066c720a669ad4f3a7a53', 'results/tables/lab03h_pos_parecer_v4_20261024/annual_overview.csv': 'a0b40220f84edab0c56827aa2b30495131747cf31759875ac89c0d58041edebc', 'results/tables/lab03i_parecer_v4_20261024/asset_liquidity.csv': 'bdd7d54335a661cddf59df909d04d41380bf4d31baa4d1ff4a00c8326bd3af16', 'results/tables/lab03i_parecer_v4_20261024/events_reproduced.csv': '4de560048a696c4d21df2ec061b90fb7f42b64a9f709b0767f314d8124c5bc90', 'results/tables/lab03j_pareceres_20261024/all_control_coefficients.csv': '081ee4d94863633a05ae6eae057528436b7c2ce2ae26836738747eb265ffda0f', 'results/tables/lab03j_pareceres_20261024/calibration_SE_diagnostic.csv': 'b92e7e52ed282f8227b48749bf83b491095ef78cfd8ef21f6d213ef5e194e736', 'results/tables/lab03j_pareceres_20261024/complete_case_audit.csv': '59ade64892fb9601276deca9eef152cf9ff4ebb0c6e9f3e5221ed5a0b4107dce', 'results/tables/lab03j_pareceres_20261024/controlled_regressions.csv': '195881ec86e0e3454007a1d7ff86d49352804188d657e9ab6c378ace8f664dad', 'results/tables/lab03j_pareceres_20261024/covariates_used.csv': 'f69cd04823780abd02c4fc09b71415e9aecf6b43e2a3d4b9f52585f9062206e1', 'results/tables/lab03j_pareceres_20261024/event_audit_83.csv': 'c3410f6ad1c3efd823454f87b707a9702d5fdbec968957f22c0054c4aba92cef', 'results/tables/lab03j_pareceres_20261024/event_capital_candidates.csv': 'acaa059e9c465b453032623de84825070bc2be3d8d35e4f1984d515b267bb048', 'results/tables/lab03j_pareceres_20261024/liquidity_change.csv': '9c4d242ae6f29b87d5582aafe744127eaf93fa5ba273c1c911dd0eb14a348df4', 'results/tables/lab03j_pareceres_20261024/succession_early_eligibility.csv': 'e4757f35d1f9a19e995fcf0c639b8926207a679b1442534b9af228b192b95312', 'results/tables/lab03j_pareceres_20261024/succession_new_flags.csv': '44c27fb60a8e55955857b7842c836798b2fbd3e2b3a363d415d6afe38fb9b84b', 'results/tables/lab03j_pareceres_20261024/succession_sensitivity.csv': '41976acbeac96729b9fc585949490df782e04b142acecb5079b97fc068d0e765', 'results/tables/lab03j_pareceres_20261024/succession_transitions.csv': '9be6cfa4293ad272d52295b49fd630a6b8a0ef4b48d85be0968a0ab8837eeeeb', 'results/tables/resposta_pareceres_20261024/inventario_117.csv': '6344e9f908dbfe2563296d9ad89e5a74e8848dca23fe3cfbade439381d0b1982', 'scripts/audit_lab03j_events.py': 'fb5512dbdf32d010325d0cc189be6402da33a6fd76fcd23d1c32342151cfae9d', 'scripts/build_lab03j_notebook.py': 'f842428eaf5f1509d227674c64429c0ffa38c5285838289a591edfd5ff333820', 'scripts/build_lab03j_paper.py': '9ebb63e05ccca37a2ae752c2e4cc66c96eac6ce8b235a16e9604276291f1d69c', 'scripts/collect_lab03j_cvm.py': 'bd00956d8341be3d823d183bce20bd33a6518c62bdb807c7aa28b1a4d90da171', 'scripts/collect_lab03j_originals.py': '303cb5b78251b317d761e0f49df46e39f8b43ae4059c17e55a44d935fdaa3f1c', 'scripts/diagnose_lab03j_calibration.py': 'b9b5764c0976e298198688e93f96a4d350b5d6dedddac680c5e807dfd90ae631', 'scripts/export_lab03j_controls.py': '36a115fcfd92af374104c68ebea00219fcd9e32f7d2cc9cd4befe86cdf2fe16f', 'scripts/extract_lab03j_turnover.py': 'a15976e0c1e747ff52df258708766dd79d543b46e8308ce35f6e1340993e21ce', 'scripts/prepare_lab03j_covariates.py': 'b3a8839397d196ce4bb8046f91a9d1dfb4cef7cd32de5ccce5f4fa2a3cfac0fd', 'scripts/run_lab03j.py': 'd06165ef218c7681cc45a54326d66b35d0cef9824c985a2db877e1ab162739ee', 'scripts/run_lab03j_batch.py': '567d5200e3629e001f31e8f1b563b830f17158a5d4600135065e28c708c86748', 'scripts/run_lab03j_successions.py': 'ce64cd290b1fd64d092fec5c687f2bfb5d9e271d3af9f38e1da5efe10076954e', 'scripts/verify_lab03j.py': 'fc58041f11bdfd34c08f909495540fece304cacf4745284324f707b32749942c', 'scripts/verify_lab03j_successions.py': '6791c8f28ae21b8d62ef064d9b016a2d987da90f467509db0b18faccf3235817', 'src/lab03j.py': 'bc4e32f66814e5bddcfec809591415b8ea765ae443093faba555768c42629788', 'tests/test_lab03j.py': '94d49d8d37813c8fa1b2ca161b43fd1245e8e8dc8c32297e780c758f5e737878', 'data/lab03j/latex_validation.json': 'd912a67b8d37f66a56430259be41c9a2b6bb864ca7f55a22a7b9c3a3f4de2511', 'data/lab03j/inputs_manifest.json': 'bca6d3c8eab5912496da2d6a0724c9fa8b0f3daa1a29b74011d65c690ecff9d2'}
for name,expected in SOURCE_PINS.items():
    target=ROOT/name;target.parent.mkdir(parents=True,exist_ok=True)
    if LOCAL_MODE:shutil.copy2(Path(os.environ['ALEM_BETA_LOCAL_SOURCE'])/name,target)
    else:urlretrieve('https://raw.githubusercontent.com/robsondf/alem-do-beta-gemini/'+SOURCE_COMMIT+'/'+name,target)
    assert digest(target)==expected,'Fonte divergente: '+name
I_ID='1L5w3uJvjTU4XiQir1BGMF3XNCWLS-_Ru';I_SHA='ff67a56f9b9cc8b91f11bcbcad311d6253fcfe95aef3f9e02303bad677b41e0e'
if LOCAL_MODE:
    I_ARCHIVE=Path(os.environ['ALEM_BETA_LOCAL_I']);assert digest(I_ARCHIVE) in [I_SHA,'2f61beb15aac8638cf92dae0eef58de8bc70cded3a6fe833519171f21f554f06']
else:
    from google.colab import auth,drive
    auth.authenticate_user();drive.mount('/content/drive')
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    credentials,_=google.auth.default();service=build('drive','v3',credentials=credentials,cache_discovery=False)
    CACHE=Path('/content/drive/MyDrive/alem_do_beta/lab03j_cache');CACHE.mkdir(parents=True,exist_ok=True)
    def existing_drive_file(file_id,target,expected):
        if target.is_file() and digest(target)==expected:return target
        part=target.with_suffix('.part')
        with part.open('wb') as stream:
            download=MediaIoBaseDownload(stream,service.files().get_media(fileId=file_id));done=False
            while not done:_,done=download.next_chunk()
        assert digest(part)==expected,'Hash de insumo divergente';part.replace(target);return target
    candidates=list(Path('/content/drive/MyDrive/alem_do_beta/lab03i_resultados').glob('*.zip'))
    I_ARCHIVE=next((q for q in candidates if digest(q)==I_SHA),None)
    if I_ARCHIVE is None:I_ARCHIVE=existing_drive_file(I_ID,CACHE/'export_I_autenticado.zip',I_SHA)
    assert digest(I_ARCHIVE)==I_SHA
I_OUTPUT=RUN/'input_I';I_OUTPUT.mkdir()
required=['market_private.csv.gz','documented_gaps.csv','gaps_preserved/returns_private.csv.gz','gaps_preserved/assets_originais_COTAHIST.csv','gaps_preserved/assets_universo_2015.csv']
with zipfile.ZipFile(I_ARCHIVE) as z:
    assert z.testzip() is None;inventory=json.loads(z.read('manifesto_exports.json'))
    for name in required:
        member='outputs_I/'+name;b=z.read(member);assert hashlib.sha256(b).hexdigest()==inventory[member],member
        target=I_OUTPUT/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(b)
official=json.loads((ROOT/'data/lab03i/b3_cache_manifest.json').read_text())
if LOCAL_MODE:B3_CACHE=Path(os.environ['ALEM_BETA_LOCAL_B3_CACHE'])
else:
    B3_CACHE=RUN/'b3_cache';B3_CACHE.mkdir();original=Path('/content/drive/MyDrive/Além do Beta/cache_b3')
    if not original.is_dir():
        first=next(Path('/content/drive/MyDrive').rglob('COTAHIST_A2015.ZIP'),None)
        if first is not None:original=first.parent
    for item in sorted(official,key=lambda x:x['name']):
        target=B3_CACHE/item['name'];candidate=original/item['name']
        if candidate.is_file() and digest(candidate)==item['sha256']:target.symlink_to(candidate)
        else:existing_drive_file(item['drive_id'],target,item['sha256'])
        print('Cache existente autenticado:',item['name'])
    shutil.copy2(ROOT/'data/lab03i/b3_cache_manifest.json',B3_CACHE/'authenticated_manifest.json')
for item in official:assert digest(B3_CACHE/item['name'])==item['sha256'],item['name']
print('Fontes:',len(SOURCE_PINS),'| export I e11caches autenticados; nenhuma cotação nova.')


## Lote completo e oráculos

Bloco21 é principal;5/63 são sensibilidades. Holm16 abrange oito modelos em duas coortes; a nova família Holm4 de liquidez/mudança e a de continuidades são distintas. Os regressores gerados são reestimados em cada sorteio síncrono. Não se reamostram ativos como iid nem se escolhe modelo/bloco pelo resultado. O preparo de porte é reproduzido e conferido antes das regressões. Aprovação de capital não é data ex;41 fatores compatíveis não são41 ajustes integralmente homologados.


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-p','test_lab03j.py','-v'],check=True)
BATCH=RUN/'batch_J';script('run_lab03j_batch.py','--i-output',I_OUTPUT,'--b3-cache',B3_CACHE,'--output',BATCH)
OUTPUT=BATCH/'outputs_J';reference=ROOT/'results/tables/lab03j_pareceres_20261024';compared=0;thresholds=0
for path in sorted(reference.glob('*.csv')):
    actual=pd.read_csv(OUTPUT/path.name,float_precision='round_trip');expected=pd.read_csv(path,float_precision='round_trip')
    pd.testing.assert_frame_equal(actual,expected,check_exact=False,atol=1e-10,rtol=1e-9);compared+=actual.size
    for column in actual.select_dtypes(include='number'):
        if column=='p' or column.startswith('p_'):
            np.testing.assert_array_equal(actual[column].le(.05),expected[column].le(.05));thresholds+=len(actual)
for name,expected in SOURCE_PINS.items():assert digest(ROOT/name)==expected,name
for name in ['independent_verification.json','succession_verification.json']:assert json.loads((OUTPUT/name).read_text())['passed']
record=dict(passed=True,environment='local_notebook_integration' if LOCAL_MODE else 'author_Colab',source_commit=SOURCE_COMMIT,source_files=len(SOURCE_PINS),compared_values=compared,p_threshold_positions=thresholds,atol=1e-10,rtol=1e-9,I_archive_sha256=digest(I_ARCHIVE),new_outer_generators=0,new_quote_downloads=0,python=sys.version,numpy=np.__version__,pandas=pd.__version__)
(OUTPUT/'notebook_comparison.json').write_text(json.dumps(record,indent=2)+chr(10));print(json.dumps(record,indent=2))


In [ ]:
display(pd.read_csv(OUTPUT/'controlled_regressions.csv').query('block==21')[['cohort','controls','N','estimate','ci_low','ci_high','p','p_holm16']])
display(pd.read_csv(OUTPUT/'liquidity_change.csv').query('block==21')[['cohort','liquidity_target','N','estimate','bootstrap_bias','p','p_holm4']])
display(pd.read_csv(OUTPUT/'event_audit_83.csv').groupby(['inside2016_2025','status']).size().rename('flags').to_frame())
display(pd.read_csv(OUTPUT/'succession_sensitivity.csv').query('block==21'))
for name in ['lab03j_assimetria_anual','lab03j_liquidez_inicial']:display(Image(filename=str(BATCH/'paper/figures/lab02'/(name+'.png'))))


## Interpretação e salvamento

Não há rejeição ajustada nos controles completos nem na associação entre mudanças. Não rejeição não demonstra equivalência ou ausência de efeito. ICs básicos são nominais e não duais dos testes centrados. A atenuação de postos é observada, mas não há teorema universal de erro nas variáveis para este desenho. Percentis de EP não validam ICs observados nem permitem excluir geradores desfavoráveis.

O ZIP inclui códigos, tabelas, figuras, bancos e insumos diários derivados para auditoria, sem os11ZIPs brutos B3. **Mantenha o export no Drive; não publique as matrizes no Git. Salve esta cópia executada do notebook e compartilhe seu link.** Homologação econômica global, tamanho exato e causalidade continuam limites de evidência, não etapas automaticamente certificadas por execução sem exceções.


In [ ]:
export=RUN/('resultados_lab03j_'+STAMP+'.zip');inventory={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for base,prefix in [(ROOT,'runtime'),(BATCH,'batch_J')]:
        for path in sorted(base.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                name=prefix+'/'+path.relative_to(base).as_posix();z.write(path,name);inventory[name]=digest(path)
    z.writestr('manifesto_exports.json',json.dumps(inventory,indent=2))
with export.open('rb') as stream:os.fsync(stream.fileno())
with zipfile.ZipFile(export) as z:
    assert z.testzip() is None;saved=json.loads(z.read('manifesto_exports.json'));assert set(saved)==set(z.namelist())-{'manifesto_exports.json'}
    for name,expected in saved.items():assert hashlib.sha256(z.read(name)).hexdigest()==expected,name
print('Export:',export.name,'| bytes:',export.stat().st_size,'| SHA256:',digest(export))
if not LOCAL_MODE:
    folder=Path('/content/drive/MyDrive/alem_do_beta/lab03j_resultados');folder.mkdir(parents=True,exist_ok=True);destination=folder/export.name;shutil.copy2(export,destination);assert digest(destination)==digest(export);print('SALVO NO DRIVE:',destination)
else:print('Integração local concluída;execução do autor no Colab ainda distinta.')
